### 1. Environment Setup & Dependencies

In [1]:
#@title Install Dependencies
!pip install -q transformers torch accelerate peft trl datasets torchao

### 2. Imports and Real Data Loading
We clone the official MassBank dataset repository and parse the first 100 real records.

In [2]:
import copy
import glob
import os
import random
import re
import torch

from datasets import Dataset
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from trl import SFTTrainer, SFTConfig

# Set random seed for reproducibility
random.seed(42)
torch.manual_seed(42)

# Clone MassBank data repository if not already present
if not os.path.exists("MassBank-data"):
    !git clone https://github.com/MassBank/MassBank-data.git

def load_real_massbank_data(repo_path="MassBank-data", max_samples=100):
    """
    Parses real MassBank .txt records and formats them for the LLM tokenization PoC.
    """
    dataset = []
    file_paths = glob.glob(f"{repo_path}/**/*.txt", recursive=True)

    for filepath in file_paths:
        if len(dataset) >= max_samples:
            break

        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                lines = f.readlines()
        except Exception:
            continue

        formula = None
        peaks = []
        in_peak_section = False

        for line in lines:
            line = line.strip()

            if line.startswith("CH$FORMULA:"):
                formula = line.split(":", 1)[1].strip()
            elif line.startswith("PK$PEAK:"):
                in_peak_section = True
            elif line == "//":
                in_peak_section = False
            elif in_peak_section:
                parts = line.split()
                if len(parts) >= 3:
                    mz = parts[0]
                    rel_int = parts[2]
                    try:
                        mz_int = str(int(float(mz)))
                        peaks.append(f"{mz_int}:{rel_int}")
                    except ValueError:
                        continue

        if formula and peaks:
            input_text = "Input: " + " ".join(peaks)
            dataset.append({
                "input_text": input_text,
                "target_formula": formula
            })

    print(f"Successfully parsed {len(dataset)} real MassBank records.")
    return dataset

real_dataset = load_real_massbank_data()
print("Example real parsed sample:", real_dataset[0])

Cloning into 'MassBank-data'...
remote: Enumerating objects: 903656, done.
remote: Counting objects: 100% (167/167), done.
remote: Compressing objects: 100% (139/139), done.
remote: Total 903656 (delta 74), reused 39 (delta 28), pack-reused 903489 (from 2)
Receiving objects: 100% (903656/903656), 517.27 MiB | 24.39 MiB/s, done.
Resolving deltas: 100% (801153/801153), done.
Updating files: 100% (144975/144975), done.
Successfully parsed 100 real MassBank records.
Example real parsed sample: {'input_text': 'Input: 170:50 183:900 184:100 253:100 267:50 365:100 383:999 384:250', 'target_formula': 'C20H32O5S'}


### 3. Custom Tokenizer Adaptation
We prepare a specialized vocabulary mapping m/z values and intensities directly to distinct tokens to avoid fracturing numeric inputs.

In [3]:
model_id = "Qwen/Qwen2.5-0.5B"
default_tokenizer = AutoTokenizer.from_pretrained(model_id)
custom_tokenizer = copy.deepcopy(default_tokenizer)

# Extract unique numerical values from real MassBank dataset
unique_numerical_strings = set([":"])
for sample in real_dataset:
    parts = sample["input_text"].replace("Input: ", "").split()
    for part in parts:
        if ":" in part:
            mz, intensity = part.split(":")
            unique_numerical_strings.add(mz)
            unique_numerical_strings.add(intensity)

custom_tokens = list(unique_numerical_strings)
num_added = custom_tokenizer.add_tokens(custom_tokens)

print("--- Custom Tokenizer Adaptation ---")
print(f"Added {num_added} unique m/z and intensity tokens.")
print(f"Default Vocab Size: {len(default_tokenizer)}")
print(f"Custom Vocab Size:  {len(custom_tokenizer)}")

config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

--- Custom Tokenizer Adaptation ---
Added 532 unique m/z and intensity tokens.
Default Vocab Size: 151665
Custom Vocab Size:  152187


### 4. Zero-Shot Baseline Evaluation

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"
base_model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map="auto")

def evaluate_model_accuracy(dataset, model, tokenizer, device, mode="Zero-Shot"):
    print(f"Starting {mode} evaluation on {len(dataset)} real samples...")
    correct_predictions = 0
    total_samples = len(dataset)
    model.eval()

    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id

    for idx, sample in enumerate(dataset):
        spectrum = sample["input_text"].replace("Input: ", "")
        target = sample["target_formula"]

        prompt = f"User: Identify the chemical compound molecular formula from the given mass spectrometry peaks (m/z:intensity).\nSpectrum: {spectrum}\nAssistant:"
        inputs = tokenizer(prompt, return_tensors="pt").to(device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=15,
                pad_token_id=pad_id,
                eos_token_id=tokenizer.eos_token_id,
                do_sample=False
            )

        full_pred_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

        prediction = ""
        if "Assistant:" in full_pred_text:
            prediction = full_pred_text.split("Assistant:")[-1].strip().split()[0]
            prediction = re.sub(r'[^A-Za-z0-9]', '', prediction)

        is_correct = prediction.lower() == target.lower()
        if is_correct:
            correct_predictions += 1

        if (idx + 1) % 20 == 0 or idx == 0:
            print(f"Sample {idx + 1}/{total_samples} | Ground Truth: {target} | Predicted: {prediction} | Match: {is_correct}")

    accuracy = (correct_predictions / total_samples) * 100
    print(f"\n{mode} Evaluation Complete! Accuracy: {accuracy:.2f}%")

evaluate_model_accuracy(real_dataset, base_model, default_tokenizer, device, mode="Zero-Shot")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

Starting Zero-Shot evaluation on 100 real samples...
Sample 1/100 | Ground Truth: C20H32O5S | Predicted: The | Match: False
Sample 20/100 | Ground Truth: C18H28O5 | Predicted: The | Match: False
Sample 40/100 | Ground Truth: C24H40O4 | Predicted: The | Match: False
Sample 60/100 | Ground Truth: C9H13N3O5 | Predicted: The | Match: False
Sample 80/100 | Ground Truth: C2H8NO4P | Predicted: The | Match: False
Sample 100/100 | Ground Truth: C27H43NO3 | Predicted: The | Match: False

Zero-Shot Evaluation Complete! Accuracy: 0.00%


### 5. Model Fine-Tuning Setup & Execution

In [6]:
# Upgrade torchao to a version compatible with peft (> 0.16.0)
!pip install -q -U torchao

# Initialize clean, non-wrapped model instance
from transformers import AutoModelForCausalLM
from datasets import Dataset
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
import torch

custom_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float32,
    device_map="auto"
)
custom_model.resize_token_embeddings(len(custom_tokenizer))

# Ensure trainable params are compatible with T4 FP32 execution parameters
for param in custom_model.parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)

custom_model.config.use_cache = False
custom_tokenizer.pad_token = custom_tokenizer.eos_token

# Prepare Dataset
def format_samples(example):
    return {
        "text": f"User: Identify the chemical compound molecular formula from the given mass spectrometry peaks (m/z:intensity).\nSpectrum: {example['input_text'].replace('Input: ', '')}\nAssistant: {example['target_formula']}"
    }

hf_dataset = Dataset.from_list(real_dataset)
formatted_dataset = hf_dataset.map(format_samples)

# QLoRA
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    modules_to_save=["embed_tokens", "lm_head"]
)

sft_config = SFTConfig(
    output_dir="./qwen_mass_spec_qlora",
    num_train_epochs=15,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    learning_rate=2e-4,
    fp16=False,
    bf16=False,
    logging_steps=5,
    save_strategy="no",
    lr_scheduler_type="constant",
    disable_tqdm=False,
    max_length=512,
    dataset_text_field="text"
)

trainer = SFTTrainer(
    model=custom_model,
    train_dataset=formatted_dataset,
    processing_class=custom_tokenizer,
    args=sft_config,
    peft_config=peft_config
)

trainer.train()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:1463: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)
W1117 11:40:57.203000 454 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Adding EOS to train dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Step,Training Loss
5,4.917988
10,3.098674
15,2.225266
20,2.132453
25,2.181132
30,1.991582
35,1.731037
40,1.625212
45,1.777612
50,1.739944


TrainOutput(global_step=375, training_loss=0.64187854997317, metrics={'train_runtime': 276.8596, 'train_samples_per_second': 5.418, 'train_steps_per_second': 1.354, 'total_flos': 650354346110976.0, 'train_loss': 0.64187854997317, 'epoch': 15.0})

### 6. Fine-Tuned Model Evaluation

In [7]:
evaluate_model_accuracy(real_dataset, custom_model, custom_tokenizer, device, mode="Post-Training")

Starting Post-Training evaluation on 100 real samples...
Sample 1/100 | Ground Truth: C20H32O5S | Predicted: C20H32O5S | Match: True
Sample 20/100 | Ground Truth: C18H28O5 | Predicted: C18H28O5 | Match: True
Sample 40/100 | Ground Truth: C24H40O4 | Predicted: C24H40O4 | Match: True
Sample 60/100 | Ground Truth: C9H13N3O5 | Predicted: C9H13N3O5 | Match: True
Sample 80/100 | Ground Truth: C2H8NO4P | Predicted: C2H8NO4P | Match: True
Sample 100/100 | Ground Truth: C27H43NO3 | Predicted: C27H43NO3 | Match: True

Post-Training Evaluation Complete! Accuracy: 97.00%
